In [1]:
import asyncio
import html
import json
import time
import urllib.request
from pathlib import Path

import numpy as np
import ipywidgets as w
from IPython.display import display

BASE = Path.home() / "queueguard"
API = "http://127.0.0.1:8000"

fc = json.loads((BASE / "artifacts/feature_contract.json").read_text())
X = np.load(BASE / "data/X_full.npy", mmap_mode="r")[fc["i2"]:]

start = w.Button(description="Start replay", button_style="success")
stop = w.Button(description="Pause", button_style="warning")
reset = w.Button(description="Reset")
speed = w.IntSlider(
    value=10, min=1, max=50, description="Batch size",
    continuous_update=False
)
status = w.HTML("<b>Ready.</b> Press Start replay.")
panel = w.HTML()
state = {
    "running": False, "task": None, "index": 0,
    "rows": [], "latencies": [], "routed": 0,
    "review": 0, "verify": 0
}

def request_api(path="/", body=None):
    data = None if body is None else json.dumps(
        body, allow_nan=False
    ).encode()
    req = urllib.request.Request(
        API + path,
        data=data,
        headers={
            "Content-Type": "application/json",
            "Connection": "close"
        }
    )
    with urllib.request.urlopen(req, timeout=10) as response:
        return json.load(response)

def card(title, value, color="#e8edf8"):
    return f"""
    <div style="background:#172238;border:1px solid #2a3953;
                border-radius:12px;padding:18px;flex:1;min-width:145px">
      <div style="color:#a8b7d0;font-size:12px">{title}</div>
      <div style="color:{color};font-size:28px;font-weight:700;
                  margin-top:8px">{value}</div>
    </div>"""

def render():
    n = state["index"]
    lat = state["latencies"]
    avg = f"{np.mean(lat):.2f} ms" if lat else "—"
    p95 = f"{np.percentile(lat, 95):.2f} ms" if lat else "—"
    bypass = 100 * (1 - state["routed"] / n) if n else 0

    cards = (
        card("PROCESSED", f"{n:,}") +
        card("ADVANCED MODEL BYPASS", f"{bypass:.1f}%", "#5ee4bc") +
        card("REVIEW RECOMMENDED", str(state["review"]), "#ff879d") +
        card("MEAN SERVER LATENCY", avg) +
        card("P95 SERVER LATENCY", p95)
    )

    recent = state["rows"][-80:]
    points = []
    for i, row in enumerate(recent):
        x = 10 + i * 780 / max(len(recent) - 1, 1)
        y = 125 - min(max(row["server_ms"], 0), 20) * 5
        points.append(f"{x:.1f},{y:.1f}")
    line = " ".join(points)

    labels = {
        "normal": ("NORMAL", "#5ee4bc"),
        "ek_dogrulama_onerisi": ("VERIFY", "#ffd27a"),
        "incelemeye_yonlendir": ("REVIEW", "#ff879d")
    }
    rows = []
    for r in reversed(state["rows"][-15:]):
        label, color = labels.get(r["decision"], ("UNKNOWN", "#a8b7d0"))
        s1 = "—" if r["s1"] is None else f'{r["s1"]:.4f}'
        s2 = "—" if r["s2"] is None else f'{r["s2"]:.4f}'
        cells = [
            str(r["id"]), s1, s2,
            html.escape(r["used"]),
            f'<span style="color:{color};font-weight:700">{label}</span>',
            f'{r["server_ms"]:.2f} ms'
        ]
        rows.append(
            "<tr>" + "".join(
                f'<td style="padding:10px;border-bottom:1px solid #26334b">{v}</td>'
                for v in cells
            ) + "</tr>"
        )

    panel.value = f"""
    <div style="background:#0d1525;color:#e8edf8;padding:26px;
                border-radius:16px;font-family:Arial,sans-serif">
      <div style="color:#7caaff;font-size:12px;letter-spacing:2px">
        TEAM AGEB · IBM Z DATATHON
      </div>
      <h1 style="color:#fff;margin:10px 0 6px">QueueGuard</h1>
      <div style="color:#a8b7d0;margin-bottom:22px">
        Two-stage fraud screening · LinuxONE API · Historical test replay
      </div>
      <div style="display:flex;gap:12px;flex-wrap:wrap">{cards}</div>

      <div style="background:#172238;padding:18px;border-radius:12px;
                  margin-top:18px">
        <b>Server latency · last 80 transactions</b>
        <div style="color:#a8b7d0;font-size:12px;margin-top:6px">
          Chart range: 0–20 ms; larger values are clipped visually.
        </div>
        <svg viewBox="0 0 800 145" style="width:100%;height:150px">
          <line x1="10" y1="125" x2="790" y2="125" stroke="#36445d"/>
          <polyline points="{line}" fill="none"
                    stroke="#73a7ff" stroke-width="2.5"/>
        </svg>
      </div>

      <div style="margin:20px 0 10px;font-weight:700">Latest transactions</div>
      <div style="overflow-x:auto">
        <table style="width:100%;text-align:left;border-collapse:collapse;
                      color:#e8edf8;font-size:13px">
          <thead style="color:#a8b7d0">
            <tr>
              <th>ID*</th><th>Light score</th><th>Advanced score</th>
              <th>Model path</th><th>Decision</th><th>Server latency</th>
            </tr>
          </thead>
          <tbody>{''.join(rows)}</tbody>
        </table>
      </div>
      <div style="color:#a8b7d0;font-size:12px;margin-top:18px">
        Additional verification: {state["verify"]} ·
        Advanced model calls: {state["routed"]}<br>
        *ID is the test replay row index. Scores are model outputs.
        Decisions are prototype recommendations, not confirmed fraud labels.
        Latencies describe this replay run, not a controlled benchmark.
      </div>
    </div>
    """

async def replay():
    start.disabled = True
    reset.disabled = True
    try:
        health = await asyncio.to_thread(request_api)
        if not health.get("ok") or health.get("mode") != "casc":
            raise RuntimeError("Expected a healthy API in casc mode.")

        while state["running"] and state["index"] < len(X):
            for _ in range(speed.value):
                if not state["running"] or state["index"] >= len(X):
                    break

                i = state["index"]
                values = np.asarray(X[i], dtype=float)
                body = {
                    "id": i,
                    "x": [float(v) if np.isfinite(v) else None for v in values]
                }
                result = await asyncio.to_thread(request_api, "/score", body)

                state["rows"].append(result)
                state["rows"] = state["rows"][-300:]
                state["latencies"].append(result["server_ms"])
                state["routed"] += int(result["routed"])
                state["review"] += int(
                    result["decision"] == "incelemeye_yonlendir"
                )
                state["verify"] += int(
                    result["decision"] == "ek_dogrulama_onerisi"
                )
                state["index"] += 1

            render()
            status.value = (
                f'<b style="color:#159b70">API connected</b> · '
                f'{state["index"]:,} / {len(X):,} test transactions · '
                f'Updated {time.strftime("%H:%M:%S")}'
            )
            await asyncio.sleep(0.25)

        if state["index"] >= len(X):
            status.value = "<b>Replay complete.</b>"
        else:
            status.value += " · Paused"
    except Exception as error:
        status.value = (
            '<b style="color:#c63753">Replay stopped:</b> ' +
            html.escape(str(error))
        )
    finally:
        state["running"] = False
        start.disabled = False
        reset.disabled = False

def on_start(_):
    if state["task"] is None or state["task"].done():
        state["running"] = True
        state["task"] = asyncio.get_running_loop().create_task(replay())

def on_stop(_):
    state["running"] = False

def on_reset(_):
    if state["task"] is not None and not state["task"].done():
        return
    state.update(
        index=0, rows=[], latencies=[],
        routed=0, review=0, verify=0
    )
    status.value = "<b>Reset.</b> Ready for a new replay."
    render()

start.on_click(on_start)
stop.on_click(on_stop)
reset.on_click(on_reset)

render()
display(w.VBox([
    w.HBox([start, stop, reset]),
    speed,
    status,
    panel
]))